In [1]:
import pandas as pd
import numpy as np
from glob import glob
import re

Arenas played in provided by Sports Reference

In [2]:
cc_shots = pd.read_csv("cc_shots.csv")     
cc_arenas = pd.read_csv("cc_arenas.csv")     

In [3]:
files = sorted(glob("cc_arena_game/cc*.csv"))
print(files)

['cc_arena_game/cc2021.csv', 'cc_arena_game/cc2022.csv', 'cc_arena_game/cc2023.csv', 'cc_arena_game/cc2024.csv', 'cc_arena_game/cc2025.csv']


In [4]:
import pandas as pd, numpy as np, re, io
from glob import glob

def _rename_sr_cols(df):
    cols = list(df.columns)
    # after 'Type' -> HA (home/@/N)
    if "Type" in cols:
        i = cols.index("Type")
        if i+1 < len(cols) and (str(cols[i+1]).startswith("Unnamed") or cols[i+1] in ("", None)):
            df = df.rename(columns={cols[i+1]: "HA"})
    # after 'SRS' -> Result (W/L)
    if "SRS" in cols:
        j = cols.index("SRS")
        if j+1 < len(cols) and (str(cols[j+1]).startswith("Unnamed") or cols[j+1] in ("", None)):
            df = df.rename(columns={cols[j+1]: "Result"})
    return df

def read_sr_schedule_file(path):
    with open(path, "r", encoding="utf-8-sig", errors="ignore") as f:
        lines = f.read().splitlines()

    # find header row (real table start)
    start = next(i for i,l in enumerate(lines)
                 if l.strip().startswith("G,Date,Time,Type") and "Arena" in l)

    # collect table until blank/HTML/"Provided by"
    body = []
    for line in lines[start:]:
        s = line.strip()
        if not s or s.startswith("Provided by") or s.startswith("<"):
            break
        body.append(line)
    if not body:
        raise ValueError(f"{path}: table block empty after trimming preface/footer.")

    df = pd.read_csv(io.StringIO("\n".join(body)), engine="python")
    return _rename_sr_cols(df)

def load_sr_home(files):
    rows = []
    for f in files:
        try:
            season = int(re.search(r"(20\d{2})", f).group(1))
            df = read_sr_schedule_file(f)
            df["HA"] = df.get("HA", "").fillna("").astype(str).str.strip()
            df["is_home"] = ~df["HA"].isin(["@", "N"])  # blank = home
            keep = df[df["is_home"] & df["Arena"].notna()].copy()
            keep["season"] = season
            rows.append(keep[["season", "Opponent", "Arena"]])
        except Exception as e:
            print(f"Skipping {f}: {e}")
    if not rows:
        raise RuntimeError("No valid home rows loaded. Check CSV contents/paths.")
    return pd.concat(rows, ignore_index=True)

# usage
sr_home = load_sr_home(files)
print(sr_home.head())


   season                 Opponent                  Arena
0    2021  Texas A&M-International   American Bank Center
1    2021              Texas State   American Bank Center
2    2021  Texas-Rio Grande Valley   American Bank Center
3    2021     Our Lady of the Lake  Dugan Wellness Center
4    2021               Paul Quinn   American Bank Center


In [5]:
sr_home

,season,Opponent,Arena
0,2021,Texas A&M-International,American Bank Center
1,2021,Texas State,American Bank Center
2,2021,Texas-Rio Grande Valley,American Bank Center
3,2021,Our Lady of the Lake,Dugan Wellness Center
4,2021,Paul Quinn,American Bank Center
...,...,...,...
67,2025,McNeese State,American Bank Center
68,2025,Incarnate Word,American Bank Center
69,2025,Houston Christian,American Bank Center
70,2025,Southeastern Louisiana,American Bank Center


In [6]:
cc_shots

,gameId,homeMarket,awayMarket,secsIntoGame,teamMarket,actionType,subType,success,side,shotDist,zones6,zones13,season,team,Year,conf,similar_team,half,primary_side,side_mismatch
0,1984957,A&M-Corpus Christi,Northwestern St.,124.0,Northwestern St.,2pt,jumpshot,False,RIGHT,20.004,mid2,rb2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False
1,1984957,A&M-Corpus Christi,Northwestern St.,352.0,Northwestern St.,2pt,jumpshot,False,RIGHT,15.822,mid2,le2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False
2,1984957,A&M-Corpus Christi,Northwestern St.,520.0,Northwestern St.,3pt,jumpshot,True,RIGHT,23.411,atb3,lw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False
3,1984957,A&M-Corpus Christi,Northwestern St.,585.0,Northwestern St.,2pt,jumpshot,True,RIGHT,17.792,mid2,rb2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False
4,1984957,A&M-Corpus Christi,Northwestern St.,665.0,Northwestern St.,3pt,jumpshot,False,RIGHT,24.284,atb3,rw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1674,2184087,A&M-Corpus Christi,CSU Bakersfield,1296.0,CSU Bakersfield,2pt,jumpshot,False,LEFT,18.312,mid2,re2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False
1675,2184087,A&M-Corpus Christi,CSU Bakersfield,1620.0,CSU Bakersfield,3pt,stepbackjumpshot,True,LEFT,23.112,c3,rc3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False
1676,2184087,A&M-Corpus Christi,CSU Bakersfield,1745.0,CSU Bakersfield,2pt,pullupjumpshot,False,LEFT,18.950,mid2,le2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False
1677,2184087,A&M-Corpus Christi,CSU Bakersfield,1851.0,CSU Bakersfield,3pt,jumpshot,False,LEFT,23.128,c3,rc3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False


In [7]:
cc_shots

,gameId,homeMarket,awayMarket,secsIntoGame,teamMarket,actionType,subType,success,side,shotDist,zones6,zones13,season,team,Year,conf,similar_team,half,primary_side,side_mismatch
0,1984957,A&M-Corpus Christi,Northwestern St.,124.0,Northwestern St.,2pt,jumpshot,False,RIGHT,20.004,mid2,rb2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False
1,1984957,A&M-Corpus Christi,Northwestern St.,352.0,Northwestern St.,2pt,jumpshot,False,RIGHT,15.822,mid2,le2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False
2,1984957,A&M-Corpus Christi,Northwestern St.,520.0,Northwestern St.,3pt,jumpshot,True,RIGHT,23.411,atb3,lw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False
3,1984957,A&M-Corpus Christi,Northwestern St.,585.0,Northwestern St.,2pt,jumpshot,True,RIGHT,17.792,mid2,rb2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False
4,1984957,A&M-Corpus Christi,Northwestern St.,665.0,Northwestern St.,3pt,jumpshot,False,RIGHT,24.284,atb3,rw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1674,2184087,A&M-Corpus Christi,CSU Bakersfield,1296.0,CSU Bakersfield,2pt,jumpshot,False,LEFT,18.312,mid2,re2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False
1675,2184087,A&M-Corpus Christi,CSU Bakersfield,1620.0,CSU Bakersfield,3pt,stepbackjumpshot,True,LEFT,23.112,c3,rc3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False
1676,2184087,A&M-Corpus Christi,CSU Bakersfield,1745.0,CSU Bakersfield,2pt,pullupjumpshot,False,LEFT,18.950,mid2,le2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False
1677,2184087,A&M-Corpus Christi,CSU Bakersfield,1851.0,CSU Bakersfield,3pt,jumpshot,False,LEFT,23.128,c3,rc3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False


In [9]:
cc_shots = cc_shots[cc_shots["awayMarket"]!="CSU Bakersfield"]

In [10]:
opp_map = {
    "Stephen F. Austin":"SFA",
    "Incarnate Word":"UIW",
    "Southeastern Louisiana":"Southeastern La.",
    "Northwestern State":"Northwestern St.",
    "McNeese State":"McNeese",
    "Texas-Rio Grande Valley":"UTRGV",
    "Nicholls State":"Nicholls",
    "Lamar":"Lamar University",
}

sr_home = sr_home.copy()
sr_home["Opponent"] = sr_home["Opponent"].replace(opp_map)

# ensure 1 row per (season, Opponent)
sr_map = sr_home[["season","Opponent","Arena"]].drop_duplicates(["season","Opponent"])

shots_with_arena = (
    cc_shots.merge(sr_map, left_on=["season","awayMarket"], right_on=["season","Opponent"], how="left")
            .drop(columns=["Opponent"])
)

shots_with_arena["Arena"].isna().sum()

np.int64(0)

In [11]:
shots_with_arena["Arena"].unique()

array(['American Bank Center', 'Dugan Wellness Center'], dtype=object)

In [12]:
cc_arenas.loc[cc_arenas.index[1], "Arena"] = "American Bank Center"

In [13]:
cc_arenas

,School,Conference,Wall Number,Side,Distance,Type of Wall,Arena,Left_Wall,Right_Wall,Left_Wall_Distance,Right_Wall_Distance,Left_Wall_Type,Right_Wall_Type
0,Texas A&M CC 1,Southland,2,Both,Close,Whole,Dugan Wellness Center,True,True,Close,Close,Whole,Whole
1,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,American Bank Center,True,False,Far,NaN,Mix,NaN


In [14]:
shots_with_arena[shots_with_arena["Arena"] == "Dugan Wellness Center"]["gameId"].unique()

array([2189338, 2551171])

In [15]:
shots_with_arena = shots_with_arena.merge(cc_arenas,on="Arena",how="left")

In [16]:
shots_with_arena

,gameId,homeMarket,awayMarket,secsIntoGame,teamMarket,actionType,subType,success,side,shotDist,...,Wall Number,Side,Distance,Type of Wall,Left_Wall,Right_Wall,Left_Wall_Distance,Right_Wall_Distance,Left_Wall_Type,Right_Wall_Type
0,1984957,A&M-Corpus Christi,Northwestern St.,124.0,Northwestern St.,2pt,jumpshot,False,RIGHT,20.004,...,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
1,1984957,A&M-Corpus Christi,Northwestern St.,352.0,Northwestern St.,2pt,jumpshot,False,RIGHT,15.822,...,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
2,1984957,A&M-Corpus Christi,Northwestern St.,520.0,Northwestern St.,3pt,jumpshot,True,RIGHT,23.411,...,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
3,1984957,A&M-Corpus Christi,Northwestern St.,585.0,Northwestern St.,2pt,jumpshot,True,RIGHT,17.792,...,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
4,1984957,A&M-Corpus Christi,Northwestern St.,665.0,Northwestern St.,3pt,jumpshot,False,RIGHT,24.284,...,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1629,1987734,A&M-Corpus Christi,Houston Christian,1906.0,Houston Christian,3pt,jumpshot,False,LEFT,25.703,...,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
1630,2190799,A&M-Corpus Christi,New Orleans,137.0,New Orleans,2pt,jumpshot,True,RIGHT,17.317,...,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
1631,2190799,A&M-Corpus Christi,New Orleans,1197.8,New Orleans,3pt,jumpshot,False,RIGHT,23.371,...,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
1632,2190799,A&M-Corpus Christi,New Orleans,1410.0,New Orleans,2pt,jumpshot,True,LEFT,18.387,...,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN


In [38]:
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", None)

In [18]:
shots_with_arena["zones13"].unique()

array(['rb2', 'le2', 'lw3', 'rw3', 'tok3', 'paint2', 're2', 'lb2', 'rc3',
       'lc3', 'heave3', 'atr2'], dtype=object)

In [24]:
correct_zones = ['le2', 'lw3', 'rw3', 'tok3', 're2', 'heave3']

In [25]:
shots_in_zones = shots_with_arena[shots_with_arena["zones13"].isin(correct_zones)]

In [26]:
shots_in_zones

,gameId,homeMarket,awayMarket,secsIntoGame,teamMarket,actionType,subType,success,side,shotDist,zones6,zones13,season,team,Year,conf,similar_team,half,primary_side,side_mismatch,Arena,School,Conference,Wall Number,Side,Distance,Type of Wall,Left_Wall,Right_Wall,Left_Wall_Distance,Right_Wall_Distance,Left_Wall_Type,Right_Wall_Type
1,1984957,A&M-Corpus Christi,Northwestern St.,352.0,Northwestern St.,2pt,jumpshot,False,RIGHT,15.822,mid2,le2,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False,American Bank Center,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
2,1984957,A&M-Corpus Christi,Northwestern St.,520.0,Northwestern St.,3pt,jumpshot,True,RIGHT,23.411,atb3,lw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False,American Bank Center,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
4,1984957,A&M-Corpus Christi,Northwestern St.,665.0,Northwestern St.,3pt,jumpshot,False,RIGHT,24.284,atb3,rw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False,American Bank Center,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
5,1984957,A&M-Corpus Christi,Northwestern St.,776.0,Northwestern St.,3pt,jumpshot,False,RIGHT,25.420,atb3,tok3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False,American Bank Center,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
7,1984957,A&M-Corpus Christi,Northwestern St.,852.0,Northwestern St.,3pt,jumpshot,False,RIGHT,23.378,atb3,tok3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,1,RIGHT,False,American Bank Center,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1627,1987734,A&M-Corpus Christi,Houston Christian,1540.0,Houston Christian,3pt,jumpshot,False,LEFT,25.458,atb3,lw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,2,LEFT,False,American Bank Center,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
1628,1987734,A&M-Corpus Christi,Houston Christian,1788.0,Houston Christian,3pt,jumpshot,False,LEFT,24.956,atb3,tok3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,2,LEFT,False,American Bank Center,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
1629,1987734,A&M-Corpus Christi,Houston Christian,1906.0,Houston Christian,3pt,jumpshot,False,LEFT,25.703,atb3,lw3,2022,A&M-Corpus Christi,2022,Slnd,Nicholls,2,LEFT,False,American Bank Center,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN
1630,2190799,A&M-Corpus Christi,New Orleans,137.0,New Orleans,2pt,jumpshot,True,RIGHT,17.317,mid2,re2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,RIGHT,False,American Bank Center,Texas A&M-Corpus Christi,Southland,1,Left,Far,Mix,True,False,Far,NaN,Mix,NaN


In [43]:
shots_with_arena[(shots_with_arena["Arena"]=="Dugan Wellness Center") & (shots_with_arena["awayMarket"]=="UIW")& (shots_with_arena["actionType"]=="2pt")]

,gameId,homeMarket,awayMarket,secsIntoGame,teamMarket,actionType,subType,success,side,shotDist,zones6,zones13,season,team,Year,conf,similar_team,half,primary_side,side_mismatch,Arena,School,Conference,Wall Number,Side,Distance,Type of Wall,Left_Wall,Right_Wall,Left_Wall_Distance,Right_Wall_Distance,Left_Wall_Type,Right_Wall_Type
84,2189338,A&M-Corpus Christi,UIW,596.0,UIW,2pt,jumpshot,True,RIGHT,20.256,mid2,lb2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,RIGHT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
86,2189338,A&M-Corpus Christi,UIW,1364.0,UIW,2pt,jumpshot,True,LEFT,19.602,mid2,le2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
87,2189338,A&M-Corpus Christi,UIW,1402.0,UIW,2pt,jumpshot,False,LEFT,15.788,mid2,le2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
89,2189338,A&M-Corpus Christi,UIW,1863.0,UIW,2pt,jumpshot,False,LEFT,20.153,mid2,lb2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
90,2189338,A&M-Corpus Christi,UIW,2153.0,UIW,2pt,jumpshot,False,LEFT,15.829,mid2,rb2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
119,2189338,A&M-Corpus Christi,UIW,280.0,UIW,2pt,jumpshot,True,RIGHT,5.291,paint2,paint2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,RIGHT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
122,2189338,A&M-Corpus Christi,UIW,1159.7,UIW,2pt,jumpshot,False,RIGHT,11.824,paint2,paint2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,RIGHT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
123,2189338,A&M-Corpus Christi,UIW,1197.8,UIW,2pt,jumpshot,False,RIGHT,8.635,paint2,paint2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,RIGHT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
125,2189338,A&M-Corpus Christi,UIW,1605.0,UIW,2pt,jumpshot,False,LEFT,8.960,paint2,paint2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
127,2189338,A&M-Corpus Christi,UIW,2025.0,UIW,2pt,jumpshot,False,LEFT,6.035,paint2,paint2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole


In [39]:
shots_in_zones[(shots_in_zones["Arena"]=="Dugan Wellness Center") & (shots_in_zones["awayMarket"]=="UIW")]

,gameId,homeMarket,awayMarket,secsIntoGame,teamMarket,actionType,subType,success,side,shotDist,zones6,zones13,season,team,Year,conf,similar_team,half,primary_side,side_mismatch,Arena,School,Conference,Wall Number,Side,Distance,Type of Wall,Left_Wall,Right_Wall,Left_Wall_Distance,Right_Wall_Distance,Left_Wall_Type,Right_Wall_Type
83,2189338,A&M-Corpus Christi,UIW,247.0,UIW,3pt,jumpshot,False,RIGHT,25.100,atb3,lw3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,RIGHT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
85,2189338,A&M-Corpus Christi,UIW,678.0,UIW,3pt,jumpshot,False,RIGHT,24.419,atb3,tok3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,RIGHT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
86,2189338,A&M-Corpus Christi,UIW,1364.0,UIW,2pt,jumpshot,True,LEFT,19.602,mid2,le2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
87,2189338,A&M-Corpus Christi,UIW,1402.0,UIW,2pt,jumpshot,False,LEFT,15.788,mid2,le2,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
88,2189338,A&M-Corpus Christi,UIW,1736.0,UIW,3pt,jumpshot,False,LEFT,25.329,atb3,tok3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
91,2189338,A&M-Corpus Christi,UIW,2305.0,UIW,3pt,jumpshot,False,LEFT,23.687,atb3,tok3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
118,2189338,A&M-Corpus Christi,UIW,102.0,UIW,3pt,jumpshot,False,RIGHT,22.697,atb3,tok3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,RIGHT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
120,2189338,A&M-Corpus Christi,UIW,453.0,UIW,3pt,jumpshot,False,RIGHT,23.789,atb3,lw3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,1,RIGHT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
124,2189338,A&M-Corpus Christi,UIW,1214.0,UIW,3pt,jumpshot,False,LEFT,24.484,atb3,lw3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole
126,2189338,A&M-Corpus Christi,UIW,1775.0,UIW,3pt,jumpshot,False,LEFT,24.197,atb3,rw3,2023,A&M-Corpus Christi,2023,Slnd,Nicholls,2,LEFT,False,Dugan Wellness Center,Texas A&M CC 1,Southland,2,Both,Close,Whole,True,True,Close,Close,Whole,Whole


In [33]:
zone_summary = (
    shots_in_zones
    .groupby(["Arena", "zones13"])
    .agg(
        attempts=("success", "count"),
        makes=("success", "sum"),
    )
    .assign(FG_pct=lambda x: x["makes"] / x["attempts"])
)


In [34]:
zone_summary

attempts  makes    FG_pct
Arena                 zones13                           
American Bank Center  heave3          7      2  0.285714
                      le2            98     24  0.244898
                      lw3           250     79  0.316000
                      re2           108     40  0.370370
                      rw3           228     69  0.302632
                      tok3          178     61  0.342697
Dugan Wellness Center le2             4      2  0.500000
                      lw3             9      1  0.111111
                      re2             2      1  0.500000
                      rw3             9      1  0.111111
                      tok3            9      1  0.111111